# Recuperación y preprocesado del caso `ImageCAS_0256`

**Objetivo:** recuperar el único caso excluido del preprocesado de `Dataset503_ImageCAS1000`, sin repetir el preprocesado de los otros 999.

El procedimiento es el siguiente:
1. Extracción del ZIP de **únicamente** la imagen y la máscara de `ImageCAS_0256`.
2. Comprobación de su integridad y geometría.
3. Reutilización de los **planes ya calculados para Dataset503**.
4. Ejecución del preprocesador de nnU-Net **solo para ese caso** mediante `DefaultPreprocessor.run_case_save`.
5. Restauración del GT original.
6. Reconstrucción de las ubicaciones de muestreo de foreground que utiliza el dataloader.
7. Verificación de que Dataset503 vuelve a contener 1000 pares preprocesados.

> **Nota:** la parte que escribe en `nnUNet_preprocessed` no debe ejecutarse mientras un entrenamiento esté leyendo o escribiendo sobre Dataset503. Las celdas de extracción y diagnóstico pueden ejecutarse en paralelo, dejando la reparación final para cuando dicho entrenamiento haya terminado o se haya detenido.


## 1. Montaje de Drive e importación de dependencias

Se utiliza el mismo entorno de Colab y la misma versión de nnU-Net que en el notebook 26.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Si nnU-Net no está instalado en la sesión, se descomenta la línea siguiente.
!pip install -q nnunetv2

import os, json, shutil, zipfile, subprocess, sys
from pathlib import Path
import numpy as np
import SimpleITK as sitk

BASE = Path('/content/drive/MyDrive/TFM_nnUNet')
ZIP_PATH = BASE / 'Dataset503_ImageCAS1000.zip'
PREP_ROOT = BASE / 'nnUNet_preprocessed'
DATASET = 'Dataset503_ImageCAS1000'
PREP_DATASET = PREP_ROOT / DATASET
CONFIG_DIR = PREP_DATASET / 'nnUNetPlans_3d_fullres'
TMP = Path('/content/reparacion_0256')
CASE = 'ImageCAS_0256'

os.environ['nnUNet_preprocessed'] = str(PREP_ROOT)
os.environ['nnUNet_results'] = str(BASE / 'nnUNet_results')

print('ZIP:', ZIP_PATH, ZIP_PATH.exists())
print('Preprocessed:', PREP_DATASET, PREP_DATASET.exists())
print('Config 3d_fullres:', CONFIG_DIR, CONFIG_DIR.exists())

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.0 MB/s eta 0:00:00
ZIP: /content/drive/MyDrive/TFM_nnUNet/Dataset503_ImageCAS1000.zip True
Pre

## 2. Localización de `0256` en el ZIP sin descomprimir los 83 GB


In [4]:
assert ZIP_PATH.exists(), f'No encuentro {ZIP_PATH}'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    matches = [n for n in z.namelist() if CASE in n]

print(f'Encontrados {len(matches)} archivos con {CASE}:')
for n in matches:
    print(' -', n)

img_members = [n for n in matches if '/imagesTr/' in n.replace('\\','/') and n.endswith(f'{CASE}_0000.nii.gz')]
seg_members = [n for n in matches if '/labelsTr/' in n.replace('\\','/') and n.endswith(f'{CASE}.nii.gz')]

assert len(img_members) == 1, f'Esperaba 1 imagen, encontré: {img_members}'
assert len(seg_members) == 1, f'Esperaba 1 máscara, encontré: {seg_members}'
print('\nOK: imagen y máscara localizadas.')

Encontrados 2 archivos con ImageCAS_0256:
 - Dataset503_ImageCAS1000/labelsTr/ImageCAS_0256.nii.gz
 - Dataset503_ImageCAS1000/imagesTr/ImageCAS_0256_0000.nii.gz

OK: imagen y máscara localizadas.


## 3. Extracción de los dos NIfTI a `/content`

Esta operación no modifica Drive ni los datos preprocesados.


In [5]:
if TMP.exists():
    shutil.rmtree(TMP)
(TMP / 'imagesTr').mkdir(parents=True)
(TMP / 'labelsTr').mkdir(parents=True)

img_path = TMP / 'imagesTr' / f'{CASE}_0000.nii.gz'
seg_path = TMP / 'labelsTr' / f'{CASE}.nii.gz'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open(img_members[0]) as src, open(img_path, 'wb') as dst:
        shutil.copyfileobj(src, dst)
    with z.open(seg_members[0]) as src, open(seg_path, 'wb') as dst:
        shutil.copyfileobj(src, dst)

print('Imagen:', img_path, round(img_path.stat().st_size/1024**2, 1), 'MB')
print('Máscara:', seg_path, round(seg_path.stat().st_size/1024**2, 1), 'MB')

Imagen: /content/reparacion_0256/imagesTr/ImageCAS_0256_0000.nii.gz 64.1 MB
Máscara: /content/reparacion_0256/labelsTr/ImageCAS_0256.nii.gz 1.9 MB


## 4. Diagnóstico del caso original

Se comprueban tamaño, spacing, origen, dirección, valores y compatibilidad imagen-máscara antes del preprocesado.


In [6]:
img = sitk.ReadImage(str(img_path))
seg = sitk.ReadImage(str(seg_path))

print('--- IMAGEN ---')
print('Size      :', img.GetSize())
print('Spacing   :', img.GetSpacing())
print('Origin    :', img.GetOrigin())
print('Direction :', img.GetDirection())
print('Pixel type:', img.GetPixelIDTypeAsString())

print('\n--- MÁSCARA ---')
print('Size      :', seg.GetSize())
print('Spacing   :', seg.GetSpacing())
print('Origin    :', seg.GetOrigin())
print('Direction :', seg.GetDirection())
print('Pixel type:', seg.GetPixelIDTypeAsString())

same_size = img.GetSize() == seg.GetSize()
same_spacing = np.allclose(img.GetSpacing(), seg.GetSpacing())
same_origin = np.allclose(img.GetOrigin(), seg.GetOrigin())
same_direction = np.allclose(img.GetDirection(), seg.GetDirection())

arr_img = sitk.GetArrayViewFromImage(img)
arr_seg = sitk.GetArrayViewFromImage(seg)

print('\n--- VALIDACIÓN ---')
print('Mismo size      :', same_size)
print('Mismo spacing   :', same_spacing)
print('Mismo origin    :', same_origin)
print('Misma direction :', same_direction)
print('Imagen min/max  :', float(np.nanmin(arr_img)), float(np.nanmax(arr_img)))
print('NaN imagen      :', int(np.isnan(arr_img).sum()))
print('Inf imagen      :', int(np.isinf(arr_img).sum()))
print('Labels máscara  :', np.unique(arr_seg).tolist())
print('Voxels FG       :', int(np.count_nonzero(arr_seg)))

assert same_size, 'ERROR: imagen y máscara tienen distinto tamaño.'
assert same_spacing, 'ERROR: imagen y máscara tienen distinto spacing.'
assert same_origin, 'ERROR: imagen y máscara tienen distinto origin.'
assert same_direction, 'ERROR: imagen y máscara tienen distinta direction.'
assert not np.isnan(arr_img).any(), 'ERROR: hay NaN en la imagen.'
assert not np.isinf(arr_img).any(), 'ERROR: hay Inf en la imagen.'
assert set(np.unique(arr_seg)).issubset({0,1}), 'ERROR: la máscara contiene etiquetas distintas de 0/1.'
print('\n✅ El NIfTI 0256 pasa las comprobaciones básicas.')

--- IMAGEN ---
Size      : (512, 512, 206)
Spacing   : (0.318359375, 0.318359375, 0.5)
Origin    : (-54.3408203125, -81.1591796875, -189.5500030517578)
Direction : (1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
Pixel type: 16-bit signed integer

--- MÁSCARA ---
Size      : (512, 512, 206)
Spacing   : (0.318359375, 0.318359375, 0.5)
Origin    : (-54.3408203125, -81.1591796875, -189.5500030517578)
Direction : (1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)
Pixel type: 64-bit float

--- VALIDACIÓN ---
Mismo size      : True
Mismo spacing   : True
Mismo origin    : True
Misma direction : True
Imagen min/max  : -1024.0 1465.0
NaN imagen      : 0
Inf imagen      : 0
Labels máscara  : [0.0, 1.0]
Voxels FG       : 163027

✅ El NIfTI 0256 pasa las comprobaciones básicas.


## 5. Verificación de los planes existentes de Dataset503

No se recalculan el fingerprint ni los planes: se reutilizan los que produjeron los otros 999 casos.


In [7]:
plans_file = PREP_DATASET / 'nnUNetPlans.json'
dataset_json_file = PREP_DATASET / 'dataset.json'

assert plans_file.exists(), f'Falta {plans_file}'
assert dataset_json_file.exists(), f'Falta {dataset_json_file}'
assert CONFIG_DIR.exists(), f'Falta {CONFIG_DIR}'

with open(dataset_json_file) as f:
    dsj = json.load(f)

print('Plans:', plans_file)
print('dataset.json:', dataset_json_file)
print('numTraining declarado:', dsj.get('numTraining'))
print('file_ending:', dsj.get('file_ending'))
print('Config dir:', CONFIG_DIR)

print('\nEstado ANTES de reparar:')
imgs = [p for p in CONFIG_DIR.glob('*.b2nd') if not p.name.endswith('_seg.b2nd')]
segs = list(CONFIG_DIR.glob('*_seg.b2nd'))
print('Imágenes .b2nd:', len(imgs))
print('Máscaras .b2nd:', len(segs))
print('0256 imagen existe:', (CONFIG_DIR / f'{CASE}.b2nd').exists())
print('0256 seg existe   :', (CONFIG_DIR / f'{CASE}_seg.b2nd').exists())

Plans: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/nnUNetPlans.json
dataset.json: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/dataset.json
numTraining declarado: 1000
file_ending: .nii.gz
Config dir: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/nnUNetPlans_3d_fullres

Estado ANTES de reparar:
Imágenes .b2nd: 999
Máscaras .b2nd: 999
0256 imagen existe: False
0256 seg existe   : False


## 6. Preprocesado de `ImageCAS_0256`

Esta celda solo debe ejecutarse cuando el entrenamiento en curso haya terminado o esté detenido.

La llamada utiliza internamente el `DefaultPreprocessor` de nnU-Net y los `nnUNetPlans` existentes. A diferencia de `nnUNetv2_preprocess -d 503`, no recorre los 1000 casos ni elimina la carpeta de configuración.


In [15]:
# Seguridad: solo se establece a True cuando no hay ningún entrenamiento de Dataset503 en ejecución.
APLICAR_REPARACION = True

if not APLICAR_REPARACION:
    print('⏸️ Reparación NO ejecutada. Poner APLICAR_REPARACION = True cuando sea seguro.')
else:
    from nnunetv2.preprocessing.preprocessors.default_preprocessor import DefaultPreprocessor
    from nnunetv2.utilities.plans_handling.plans_handler import PlansManager

    plans_manager = PlansManager(str(plans_file))
    configuration_manager = plans_manager.get_configuration('3d_fullres')
    pp = DefaultPreprocessor(verbose=True)

    out_truncated = str(CONFIG_DIR / CASE)
    pp.run_case_save(
        out_truncated,
        [str(img_path)],
        str(seg_path),
        plans_manager,
        configuration_manager,
        str(dataset_json_file)
    )

    print('\n✅ Preprocesado individual terminado.')
    print('Imagen:', (CONFIG_DIR / f'{CASE}.b2nd').exists())
    print('Seg    :', (CONFIG_DIR / f'{CASE}_seg.b2nd').exists())
    print('PKL    :', (CONFIG_DIR / f'{CASE}.pkl').exists())

/content/reparacion_0256/labelsTr/ImageCAS_0256.nii.gz
old shape: (206, 512, 512), new_shape: [206 466 466], old_spacing: [np.float64(0.5), np.float64(0.318359375), np.float64(0.318359375)], new_spacing: [0.5, 0.349609375, 0.349609375], fn_data: functools.partial(<function resample_data_or_seg_to_shape at 0x7aac843d9760>, is_seg=False, order=3, order_z=0, force_separate_z=None)
1 10000

✅ Preprocesado individual terminado.
Imagen: True
Seg    : True
PKL    : True


## 7. Restauración de `gt_segmentations/ImageCAS_0256.nii.gz`

En el notebook 26 se eliminó este GT al excluir temporalmente el caso. Aquí se restaura a partir de la máscara original.


In [16]:
if APLICAR_REPARACION:
    gt_dir = PREP_DATASET / 'gt_segmentations'
    gt_dir.mkdir(exist_ok=True)
    gt_dest = gt_dir / f'{CASE}.nii.gz'
    shutil.copy2(seg_path, gt_dest)
    print('✅ GT restaurado:', gt_dest, gt_dest.exists())
else:
    print('⏸️ Omitido porque APLICAR_REPARACION=False')

✅ GT restaurado: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed/Dataset503_ImageCAS1000/gt_segmentations/ImageCAS_0256.nii.gz True


## 8. Reconstrucción de las ubicaciones de muestreo de foreground

Las versiones recientes de nnU-Net almacenan las coordenadas usadas en el *foreground oversampling* en un almacén compartido dentro de la configuración preprocesada. `run_case_save` genera el caso (`.b2nd` y `.pkl`) pero no actualiza por sí solo ese almacén, por lo que se emplea la utilidad oficial de reconstrucción, `nnUNetv2_extract_sampling_locations`, cuya documentación la describe precisamente para este fin.

El procedimiento no repite el remuestreo ni la normalización de los 1000 volúmenes: opera sobre las segmentaciones ya preprocesadas.


In [18]:
import shutil
import importlib.util

print("nnUNet instalado:",
      importlib.util.find_spec("nnunetv2") is not None)

print("nnUNetv2_train:",
      shutil.which("nnUNetv2_train"))

print("extract_sampling_locations:",
      shutil.which("nnUNetv2_extract_sampling_locations"))

nnUNet instalado: True
nnUNetv2_train: /usr/local/bin/nnUNetv2_train
extract_sampling_locations: None


In [19]:
from pathlib import Path

PREP = Path(
    "/content/drive/MyDrive/TFM_nnUNet/"
    "nnUNet_preprocessed/Dataset503_ImageCAS1000"
)

print("=== Archivos de 0256 actualmente en PREPROCESSED ===")

for p in sorted(PREP.rglob("*0256*")):
    print(p.relative_to(PREP), " | ", round(p.stat().st_size / 1024**2, 2), "MB")

print("\n=== Conteo dentro de 3d_fullres ===")

CONFIG = PREP / "nnUNetPlans_3d_fullres"

b2nd = sorted(CONFIG.glob("ImageCAS_*.b2nd"))
pkl  = sorted(CONFIG.glob("ImageCAS_*.pkl"))

print("Todos los .b2nd:", len(b2nd))
print("Todos los .pkl :", len(pkl))

print("\n0256:")
for nombre in [
    "ImageCAS_0256.b2nd",
    "ImageCAS_0256_seg.b2nd",
    "ImageCAS_0256.pkl"
]:
    p = CONFIG / nombre
    print(nombre, "→", p.exists(),
          f"{p.stat().st_size / 1024**2:.2f} MB" if p.exists() else "")

=== Archivos de 0256 actualmente en PREPROCESSED ===
gt_segmentations/ImageCAS_0256.nii.gz  |  1.89 MB
nnUNetPlans_3d_fullres/ImageCAS_0256.b2nd  |  107.75 MB
nnUNetPlans_3d_fullres/ImageCAS_0256.pkl  |  0.31 MB
nnUNetPlans_3d_fullres/ImageCAS_0256_seg.b2nd  |  0.21 MB

=== Conteo dentro de 3d_fullres ===
Todos los .b2nd: 2000
Todos los .pkl : 1000

0256:
ImageCAS_0256.b2nd → True 107.75 MB
ImageCAS_0256_seg.b2nd → True 0.21 MB
ImageCAS_0256.pkl → True 0.31 MB


In [20]:
from pathlib import Path
import re

PREP = Path(
    "/content/drive/MyDrive/TFM_nnUNet/"
    "nnUNet_preprocessed/Dataset503_ImageCAS1000"
)

CONFIG = PREP / "nnUNetPlans_3d_fullres"
GT = PREP / "gt_segmentations"

# Imagen preprocesada
imgs = {
    re.search(r"ImageCAS_(\d{4})\.b2nd$", p.name).group(1)
    for p in CONFIG.glob("ImageCAS_*.b2nd")
    if re.search(r"ImageCAS_(\d{4})\.b2nd$", p.name)
}

# Segmentación preprocesada
segs = {
    re.search(r"ImageCAS_(\d{4})_seg\.b2nd$", p.name).group(1)
    for p in CONFIG.glob("ImageCAS_*_seg.b2nd")
    if re.search(r"ImageCAS_(\d{4})_seg\.b2nd$", p.name)
}

# Propiedades
pkls = {
    re.search(r"ImageCAS_(\d{4})\.pkl$", p.name).group(1)
    for p in CONFIG.glob("ImageCAS_*.pkl")
    if re.search(r"ImageCAS_(\d{4})\.pkl$", p.name)
}

# Ground truth original
gts = {
    re.search(r"ImageCAS_(\d{4})\.nii\.gz$", p.name).group(1)
    for p in GT.glob("ImageCAS_*.nii.gz")
    if re.search(r"ImageCAS_(\d{4})\.nii\.gz$", p.name)
}

esperados = {f"{i:04d}" for i in range(1, 1001)}

print("========== DATASET503 ==========")
print("Imágenes preprocesadas :", len(imgs))
print("Segmentaciones         :", len(segs))
print("Properties (.pkl)      :", len(pkls))
print("GT originales          :", len(gts))

print("\n========== FALTANTES ==========")
print("Imagen:", sorted(esperados - imgs))
print("Seg:   ", sorted(esperados - segs))
print("PKL:   ", sorted(esperados - pkls))
print("GT:    ", sorted(esperados - gts))

ok = imgs == segs == pkls == gts == esperados

print("\n================================")
if ok:
    print("✅ DATASET503 COMPLETO: 1000/1000")
else:
    print("❌ TODAVÍA HAY INCONSISTENCIAS")
print("================================")

========== DATASET503 ==========
Imágenes preprocesadas : 1000
Segmentaciones         : 1000
Properties (.pkl)      : 1000
GT originales          : 1000

========== FALTANTES ==========
Imagen: []
Seg:    []
PKL:    []
GT:     []

✅ DATASET503 COMPLETO: 1000/1000


## 9. Verificación final: 1000 pares

La comprobación exige que estén presentes los tres componentes de cada caso (`.b2nd`, `_seg.b2nd` y `.pkl`) y que los recuentos sean 1000/1000.


In [21]:
imgs = [p for p in CONFIG_DIR.glob('*.b2nd') if not p.name.endswith('_seg.b2nd')]
segs = list(CONFIG_DIR.glob('*_seg.b2nd'))
pkls = list(CONFIG_DIR.glob('*.pkl'))
gts = list((PREP_DATASET / 'gt_segmentations').glob('*.nii.gz'))

print('Imágenes preprocesadas :', len(imgs))
print('Máscaras preprocesadas :', len(segs))
print('PKL propiedades        :', len(pkls))
print('GT originales          :', len(gts))
print('\n0256:')
print(' image:', (CONFIG_DIR / f'{CASE}.b2nd').exists())
print(' seg  :', (CONFIG_DIR / f'{CASE}_seg.b2nd').exists())
print(' pkl  :', (CONFIG_DIR / f'{CASE}.pkl').exists())
print(' GT   :', (PREP_DATASET / 'gt_segmentations' / f'{CASE}.nii.gz').exists())

if APLICAR_REPARACION:
    assert len(imgs) == 1000, f'Esperaba 1000 imágenes, hay {len(imgs)}'
    assert len(segs) == 1000, f'Esperaba 1000 máscaras, hay {len(segs)}'
    assert (CONFIG_DIR / f'{CASE}.b2nd').exists()
    assert (CONFIG_DIR / f'{CASE}_seg.b2nd').exists()
    assert (CONFIG_DIR / f'{CASE}.pkl').exists()
    assert (PREP_DATASET / 'gt_segmentations' / f'{CASE}.nii.gz').exists()
    print('\n🎉 Dataset503 reparado: ImageCAS_0256 vuelve a estar disponible.')
else:
    print('\nDiagnóstico completado. Para reparar, volver a la celda 6 y poner APLICAR_REPARACION=True.')

Imágenes preprocesadas : 1000
Máscaras preprocesadas : 1000
PKL propiedades        : 1000
GT originales          : 1000

0256:
 image: True
 seg  : True
 pkl  : True
 GT   : True

🎉 Dataset503 reparado: ImageCAS_0256 vuelve a estar disponible.


In [22]:
splits_path = PREP_DATASET / "splits_final.json"
print("splits_final.json existe:", splits_path.exists())
if splits_path.exists():
    import json
    splits = json.loads(splits_path.read_text())
    n_train_fold0 = len(splits[0]["train"])
    n_val_fold0 = len(splits[0]["val"])
    print(f"Fold 0 -> train: {n_train_fold0}, val: {n_val_fold0}, total: {n_train_fold0 + n_val_fold0}")
    print("0256 en train:", "ImageCAS_0256" in splits[0]["train"])
    print("0256 en val:", "ImageCAS_0256" in splits[0]["val"])

splits_final.json existe: True
Fold 0 -> train: 799, val: 200, total: 999
0256 en train: False
0256 en val: False


In [23]:
if splits_path.exists():
    splits_path.unlink()
    print("splits_final.json borrado -- nnUNet generara uno nuevo con los 1000 casos al lanzar el entrenamiento.")

splits_final.json borrado -- nnUNet generara uno nuevo con los 1000 casos al lanzar el entrenamiento.


## 10. Consideraciones posteriores a la reparación

Antes del entrenamiento definitivo se revisa `splits_final.json`. El entrenamiento preliminar sobre 999 casos pudo generar un split automático de 5 folds basado en esos 999 identificadores, por lo que dicho archivo no debe reutilizarse sin revisión tras reincorporar 0256.

La recuperación de 0256 y la elección entre el split automático de nnU-Net o la partición de ImageCAS son decisiones independientes.
